Neighbourhood crime rate        

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import scipy
import PIL
import requests

In [7]:
import pandas as pd
import plotly.graph_objects as go

# 1. Load and Clean the dataset
file_path = 'neighbourhood-crime-rates - 4326.csv'
df = pd.read_csv(file_path, low_memory=False)

# Prepare years and columns
years = list(range(2014, 2026))
rate_columns = [f'BIKETHEFT_RATE_{year}' for year in years]

# Ensure numeric data and clean missing values
for col in rate_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=['AREA_NAME'])
all_neighbourhoods = sorted(df['AREA_NAME'].unique().tolist())

# Identify Top 10 for the default view
top_10_names = df.nlargest(10, 'BIKETHEFT_RATE_2025')['AREA_NAME'].values

# 2. Create the Figure
fig = go.Figure()

# Add a trace for every single neighbourhood
for hood in all_neighbourhoods:
    hood_data = df[df['AREA_NAME'] == hood]
    
    fig.add_trace(
        go.Scatter(
            x=years,
            y=hood_data[rate_columns].values.flatten(),
            name=hood,
            mode='lines+markers',
            # Only show the Top 10 by default
            visible=True if hood in top_10_names else False 
        )
    )

# 3. Build the Dropdown Menu
dropdown_buttons = [
    # Option 1: Show the Top 10 (Default)
    dict(
        label="Top 10 Neighbourhoods",
        method="update",
        args=[{"visible": [h in top_10_names for h in all_neighbourhoods]},
              {"title": "Bike Theft Rate: Top 10 Neighbourhoods (2025)"}]
    )
]

# Add options for every individual city
for hood in all_neighbourhoods:
    dropdown_buttons.append(
        dict(
            label=hood,
            method="update",
            args=[{"visible": [h == hood for h in all_neighbourhoods]},
                  {"title": f"Bike Theft Rate Trend: {hood}"}]
        )
    )

# 4. Update Layout with Dropdown
fig.update_layout(
    updatemenus=[
        dict(
            buttons=dropdown_buttons,
            direction="down",
            showactive=True,
            x=0.0,
            y=1.15
        )
    ],
    title="Bike Theft Rate Trend (2014-2025)",
    xaxis_title="Year",
    yaxis_title="Rate per 100,000 residents",
    template="plotly_white",
    hovermode="x unified"
)

# Save as HTML to keep interactivity
fig.write_html("interactive_bike_theft_graph.html")
fig.show()